# RAG Multi-LLM Benchmark + LLM-as-Judge

This notebook is designed as a guided lab for students who are new to Retrieval-Augmented Generation (RAG), model benchmarking, and automated evaluation.

What you will learn:
- What a RAG pipeline is and why it reduces hallucinations compared with plain prompting.
- How to test several LLMs on exactly the same retrieval context for fair comparison.
- How to save experiment results in a structured way so your analysis is reproducible.
- How to use a separate LLM as a judge with an explicit scoring rubric.

What this notebook uses as inputs (already created in a previous notebook and also available at https://osf.io/atfwv):
- `campaign_instances.csv`: campaign metadata and campaign text.
- `campaign_embeddings_bert_mini.npy`: vector embeddings for each campaign.

At the end, you will produce:
- One table with model answers for each benchmark question.
- One table with judge scores per answer.
- A per-question winner table and an overall leaderboard.

Important teaching note:
- An LLM judge is useful, but it is not perfect ground truth. Treat judge scores as decision support, not absolute truth.

## 1) Install Dependencies (Optional)

Run the next cell only if your environment does not already have the required packages.

Why this matters:
- Different environments (local machine, Colab, lab server) often start with different packages installed.
- A single install cell makes the notebook portable and easier to troubleshoot in class.

Tip for beginners:
- If package installation asks for a runtime restart, restart and then run all cells from the top.

In [ ]:
# %pip install -q pandas numpy torch transformers tqdm python-dotenv requests openai anthropic google-generativeai osfclient plotly scipy

## 2) Imports

This section loads all Python libraries used in the notebook.

Conceptual grouping of imports:
- Data handling: `pandas`, `numpy`, `json`, `pathlib`.
- Embedding and retrieval: `torch`, `transformers`.
- API communication: `requests`, provider SDKs loaded later.
- Configuration: `dotenv` for local environment variables and secret management.
- Progress tracking: `tqdm` for long loops.

Why this helps students:
- Keeping imports together at the start makes dependency errors appear early and easier to fix.

In [19]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from dotenv import load_dotenv
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer
from osfclient import OSF

## 3) Secrets and Runtime Config

This step reads API keys and runtime settings. It supports two secret sources:
- Local `.env` file (typical on your own machine).
- Google Colab secrets (typical in browser-based sessions).

What this cell does:
- Detects which provider keys are actually available.
- Loads shared hyperparameters used later in generation and retrieval (for example `TOP_K`, temperature, and output length).

Important design choice in this notebook:
- **Model names are not loaded from `.env`.**
- Model selection is done in the next standalone code cell so you can change models directly inside the notebook during class.

Why this is important for benchmarking fairness:
- All models should be tested under consistent settings whenever possible.
- Centralized config reduces accidental differences between model runs.

In [30]:
def _find_dotenv_path(start_dir: Path = None) -> Path | None:
    start_dir = start_dir or Path.cwd()
    for d in [start_dir, *start_dir.parents]:
        candidate = d / ".env"
        if candidate.exists():
            return candidate
    return None

dotenv_path = _find_dotenv_path()
if dotenv_path is not None:
    # override=True avoids stale empty runtime env vars masking .env values.
    load_dotenv(dotenv_path=dotenv_path, override=True)
    print(f"Loaded .env")
else:
    load_dotenv(override=True)
    print("No .env found by parent search; tried default dotenv discovery.")

def _read_colab_secret(secret_name: str):
    try:
        from google.colab import userdata  # type: ignore
    except Exception:
        return None

    try:
        value = userdata.get(secret_name)
        return value if value else None
    except Exception:
        return None

def get_secret(secret_name: str, default: str = "") -> str:
    env_val = os.getenv(secret_name)
    if env_val:
        return env_val
    colab_val = _read_colab_secret(secret_name)
    if colab_val:
        return colab_val
    return default

API_KEYS = {
    "OPENAI_API_KEY": get_secret("OPENAI_API_KEY"),
    "GEMINI_API_KEY": get_secret("GEMINI_API_KEY"),
    "ANTHROPIC_API_KEY": get_secret("ANTHROPIC_API_KEY"),
    "HUGGINGFACE_API_KEY": get_secret("HUGGINGFACE_API_KEY"),
}

for key_name, key_value in API_KEYS.items():
    print(f"{key_name}: {'SET' if key_value else 'MISSING'}")

TOP_K = int(os.getenv("TOP_K", "5"))
MAX_CONTEXT_CHARS = int(os.getenv("MAX_CONTEXT_CHARS", "7000"))
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.2"))
MAX_OUTPUT_TOKENS = int(os.getenv("MAX_OUTPUT_TOKENS", "500"))

Loaded .env
OPENAI_API_KEY: MISSING
GEMINI_API_KEY: SET
ANTHROPIC_API_KEY: MISSING
HUGGINGFACE_API_KEY: SET


## 3.1) Model Selection 

In this notebook cell you select:
- Exactly **2 models** that will answer the benchmark questions.
- Exactly **1 judge model** that will score those answers.

In [64]:
# Single source of truth for model selection in this notebook.
# Select exactly 2 models for answering benchmark questions.
# Note: this runtime currently cannot resolve api-inference.huggingface.co DNS,
# so both answer models are configured on Gemini for reliable execution.
BENCHMARK_MODELS = [
    {
        "llm_id": "gemini_answerer_a",
        "provider": "gemini",
        "model": "gemma-4-31b-it",  # CHANGE HERE IF NECESSARY
        "api_key_name": "GEMINI_API_KEY",
    },
    {
        "llm_id": "gemini_answerer_b",
        "provider": "gemini",
        "model": "gemma-4-26b-a4b-it",  # CHANGE HERE IF NECESSARY
        "api_key_name": "GEMINI_API_KEY",
    },
]

# Select exactly 1 model for acting as judge.
JUDGE_MODEL_CONFIG = {
    "provider": "gemini",
    "model": "gemini-3.1-flash-lite",  # CHANGE HERE IF NECESSARY
    "api_key_name": "GEMINI_API_KEY",
}

print("Selected answer models:")
for m in BENCHMARK_MODELS:
    print(f"- {m['llm_id']}: {m['provider']} / {m['model']}")

print("Selected judge model:")
print(f"- {JUDGE_MODEL_CONFIG['provider']} / {JUDGE_MODEL_CONFIG['model']}")

Selected answer models:
- gemini_answerer_a: gemini / gemma-4-31b-it
- gemini_answerer_b: gemini / gemma-4-26b-a4b-it
Selected judge model:
- gemini / gemini-3.1-flash-lite


## 4) Load Existing Campaign Artifacts

This section loads the two required data artifacts with a robust two-step strategy:
- First try OSF (project download).
- If OSF is unavailable or files are missing, fall back to local files on disk.

Why this is useful in teaching settings:
- Students can run the notebook from different machines without manually copying files first.
- The same notebook works both online (OSF) and offline (local fallback).

What gets loaded:
- `campaign_instances.csv`: campaign text and metadata.
- `campaign_embeddings_bert_mini.npy`: embedding matrix where each row matches one campaign.

Key concept for students:
- Row alignment must be preserved: row `i` in the CSV must correspond to row `i` in the embedding matrix.
- If alignment is broken, retrieval quality becomes unreliable.

Quick sanity checks in the code output:
- Number of rows in the table.
- Embedding matrix shape (rows x dimensions).
- Whether each file came from OSF or local fallback.

In [49]:
DATA_TABLE_PATH = Path("campaign_instances.csv")
EMBED_PATH = Path("campaign_embeddings_bert_mini.npy")

OSF_PROJECT_ID = os.getenv("OSF_PROJECT_ID", "atfwv")
OSF_STORAGE = os.getenv("OSF_STORAGE", "osfstorage")

def try_download_osf_file(project_id: str, storage_name: str, filename: str, dest_path: Path) -> bool:
    try:
        from osfclient import OSF
    except Exception as e:
        print(f"OSF download skipped for {filename}: osfclient not available ({type(e).__name__}).")
        return False

    try:
        osf = OSF()
        project = osf.project(project_id)
        storage = project.storage(storage_name)

        for file_node in storage.files:
            if file_node.name == filename:
                with dest_path.open("wb") as local_file:
                    file_node.write_to(local_file)
                print(f"Downloaded from OSF: {filename}")
                return True

        print(f"OSF file not found: {filename} in project {project_id}/{storage_name}")
        return False
    except Exception as e:
        print(f"OSF download failed for {filename}: {type(e).__name__}: {e}")
        return False

def ensure_file_available(path: Path, project_id: str, storage_name: str) -> str:
    downloaded = try_download_osf_file(project_id, storage_name, path.name, path)
    if downloaded:
        return "osf"
    if path.exists():
        print(f"Using local file: {path.name}")
        return "local"
    raise FileNotFoundError(
        f"Could not load {path.name}: OSF download failed and local file not found at {path.resolve()}"
    )

source_table = ensure_file_available(DATA_TABLE_PATH, OSF_PROJECT_ID, OSF_STORAGE)
source_embed = ensure_file_available(EMBED_PATH, OSF_PROJECT_ID, OSF_STORAGE)

df = pd.read_csv(DATA_TABLE_PATH)
embeddings = np.load(EMBED_PATH)

print("Source (table):", source_table)
print("Source (embeddings):", source_embed)
print("Campaign rows:", len(df))
print("Embeddings shape:", embeddings.shape)
df.head()

100%|██████████| 56.3k/56.3k [00:00<00:00, 3.47Mbytes/s]


Downloaded from OSF: campaign_instances.csv


100%|██████████| 58.5k/58.5k [00:00<00:00, 2.58Mbytes/s]

Downloaded from OSF: campaign_embeddings_bert_mini.npy
Source (table): osf
Source (embeddings): osf
Campaign rows: 57
Embeddings shape: (57, 256)


,url,title,campaign_instance_id,campaign_family_key,instance_number,instance_count_in_family,campaign_text
0,https://www.swiss-solidarity.org/campaigns/aid...,Aid following the earthquake in Nepal,aid following the earthquake in nepal||aid-fol...,aid following the earthquake in nepal||aid-fol...,1,1,Aid following the earthquake in Nepal We need ...
1,https://www.swiss-solidarity.org/campaigns/aid...,Aid for Afghanistan,aid for afghanistan||aid-for-afghanistan__inst...,aid for afghanistan||aid-for-afghanistan,1,1,"Aid for Afghanistan We need you Your donation,..."
2,https://www.swiss-solidarity.org/campaigns/aid...,Aid in response to disasters and crises in Swi...,aid in response to disasters and crises in swi...,aid in response to disasters and crises in swi...,1,1,Aid in response to disasters and crises in Swi...
3,https://www.swiss-solidarity.org/campaigns/asi...,Asia 2009,asia 2009||asia__instance_1,asia 2009||asia,1,1,"Asia 2009 We need you Your donation, our actio..."
4,https://www.swiss-solidarity.org/campaigns/chi...,Child Protection in Switzerland,child protection in switzerland||child-protect...,child protection in switzerland||child-protect...,1,1,Child Protection in Switzerland We need you Yo...


## 5) Query Embedding and Retrieval Functions

This section is mostly about **definition**, not full experiment execution.

How retrieval works in this notebook:
- Convert the user question into an embedding vector using the same embedding model used for documents.
- Compute cosine similarity between the question vector and every campaign vector.
- Select the top-k most similar campaigns as evidence context.

What the next code cell **defines**:
- `mean_pool(...)` for token-to-sentence pooling.
- `embed_texts(...)` to embed batches of text.
- `embed_one_text(...)` to embed a single query.
- `retrieve_top_k(...)` to rank campaign documents for a question.

What the next code cell **executes immediately**:
- The final line runs a small demo query:
  `retrieve_top_k("list all campaigns related to earthquakes", k=5)`
- This is a quick sanity check, not the full benchmark.

Where full experiment execution happens:
- The complete multi-question, multi-model run happens later in Section 9.

Why we use the same embedding model for queries and documents:
- Query/document vectors are only comparable if they live in the same embedding space.

In [37]:
EMBEDDING_MODEL_NAME = "prajjwal1/bert-mini"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
embedding_model = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME).to(device)
embedding_model.eval()

def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts(texts, batch_size: int = 16, max_length: int = 256):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = embedding_model(**encoded)
            vecs = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            vecs = torch.nn.functional.normalize(vecs, p=2, dim=1)

        all_vecs.append(vecs.cpu().numpy())

    return np.vstack(all_vecs)

def embed_one_text(text: str, max_length: int = 256) -> np.ndarray:
    vec = embed_texts([text], batch_size=1, max_length=max_length)
    return vec[0]

def retrieve_top_k(query: str, k: int = 5) -> pd.DataFrame:
    query_vec = embed_one_text(query).astype(np.float32, copy=False)
    doc_vecs = embeddings.astype(np.float32, copy=False)

    q = torch.from_numpy(query_vec)
    d = torch.from_numpy(doc_vecs)
    scores = torch.nn.functional.cosine_similarity(d, q.unsqueeze(0), dim=1, eps=1e-8)
    scores = scores.cpu().numpy()

    top_idx = np.argsort(scores)[::-1][:k]
    out = df.iloc[top_idx].copy()
    out["similarity"] = scores[top_idx]
    return out.reset_index(drop=True)

retrieve_top_k("list all campaigns related to earthquakes", k=5)[["title", "url", "similarity"]]

Using device: cpu


,title,url,similarity
0,Jeder Rappen zählt 2009 – 2018,https://www.swiss-solidarity.org/campaigns/jed...,0.556717
1,Natural disasters in Switzerland: Blatten,https://www.swiss-solidarity.org/campaigns/nat...,0.552170
2,Haiti: aid following the earthquake,https://www.swiss-solidarity.org/campaigns/hai...,0.550747
3,Severe weather events in Switzerland in 2024,https://www.swiss-solidarity.org/campaigns/sev...,0.550446
4,Aid following the earthquake in Nepal,https://www.swiss-solidarity.org/campaigns/aid...,0.548490


## 6) Multi-Provider LLM Generation

This section is also primarily **definition** of reusable helper functions.

Educational goal:
- You can compare models fairly without rewriting your pipeline for each provider.

What the next code cell **defines**:
- Provider-specific generation functions for OpenAI, Gemini, Anthropic, and Hugging Face.
- `generate_answer(...)` as a unified dispatcher.

What the next code cell **does not execute yet**:
- It does not run the benchmark loop.
- It does not generate all answers for all questions.

Where execution happens:
- Actual batched generation is executed in Section 9 when the loop iterates through questions and active models.

Why this design is useful:
- Easier experimentation (swap model/provider in one line).
- Cleaner debugging (provider-specific errors stay isolated).
- Reproducible benchmarking across different model ecosystems.

In [58]:
import random
import time

def _is_retryable_error(exc: Exception) -> bool:
    msg = str(exc).lower()
    retry_markers = [
        "429",
        "500",
        "502",
        "503",
        "504",
        "timeout",
        "timed out",
        "connection",
        "temporarily unavailable",
        "service unavailable",
        "internal error",
    ]
    return any(marker in msg for marker in retry_markers)

def _run_with_retries(call_fn, max_retries: int = 4, base_delay: float = 1.2):
    last_exc = None
    for attempt in range(max_retries + 1):
        try:
            return call_fn()
        except Exception as e:
            last_exc = e
            if attempt >= max_retries or not _is_retryable_error(e):
                raise
            sleep_s = base_delay * (2 ** attempt) + random.uniform(0.0, 0.4)
            print(
                f"Retryable error ({type(e).__name__}) on attempt {attempt + 1}/{max_retries + 1}. "
                f"Sleeping {sleep_s:.1f}s..."
            )
            time.sleep(sleep_s)
    raise RuntimeError(f"Request failed after retries: {last_exc}")

def _allowed_models_from_selection() -> set[str]:
    allowed = set()
    if "BENCHMARK_MODELS" in globals():
        for cfg in BENCHMARK_MODELS:
            model_name = str(cfg.get("model", "")).strip()
            if model_name:
                allowed.add(model_name)
    if "JUDGE_MODEL_CONFIG" in globals():
        judge_model = str(JUDGE_MODEL_CONFIG.get("model", "")).strip()
        if judge_model:
            allowed.add(judge_model)
    return allowed

def generate_openai(prompt: str, model: str, temperature: float = 0.2, max_tokens: int = 500) -> str:
    from openai import OpenAI

    api_key = API_KEYS["OPENAI_API_KEY"]
    if not api_key:
        raise ValueError("OPENAI_API_KEY is missing.")

    client = OpenAI(api_key=api_key)

    def _call():
        response = client.responses.create(
            model=model,
            input=prompt,
            temperature=temperature,
            max_output_tokens=max_tokens,
        )
        return response.output_text

    return _run_with_retries(_call)

def generate_gemini(prompt: str, model: str, temperature: float = 0.2, max_tokens: int = 500) -> str:
    import google.generativeai as genai

    api_key = API_KEYS["GEMINI_API_KEY"]
    if not api_key:
        raise ValueError("GEMINI_API_KEY is missing.")

    genai.configure(api_key=api_key)

    def _call():
        gen_model = genai.GenerativeModel(model_name=model)
        response = gen_model.generate_content(
            prompt,
            generation_config={
                "temperature": temperature,
                "max_output_tokens": max_tokens,
            },
        )
        return response.text if hasattr(response, "text") else str(response)

    return _run_with_retries(_call)

def generate_anthropic(prompt: str, model: str, temperature: float = 0.2, max_tokens: int = 500) -> str:
    import anthropic

    api_key = API_KEYS["ANTHROPIC_API_KEY"]
    if not api_key:
        raise ValueError("ANTHROPIC_API_KEY is missing.")

    client = anthropic.Anthropic(api_key=api_key)

    def _call():
        message = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role": "user", "content": prompt}],
        )
        texts = [block.text for block in message.content if getattr(block, "type", None) == "text"]
        return "\n".join(texts).strip()

    return _run_with_retries(_call)

def generate_huggingface(prompt: str, model: str, temperature: float = 0.2, max_tokens: int = 500) -> str:
    api_key = API_KEYS["HUGGINGFACE_API_KEY"]
    if not api_key:
        raise ValueError("HUGGINGFACE_API_KEY is missing.")

    url = f"https://api-inference.huggingface.co/models/{model}"
    headers = {"Authorization": f"Bearer {api_key}"}
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": max_tokens,
            "temperature": temperature,
            "return_full_text": False,
        },
        "options": {
            "wait_for_model": True,
            "use_cache": False,
        },
    }

    def _call():
        response = requests.post(url, headers=headers, json=payload, timeout=180)
        response.raise_for_status()
        data = response.json()

        if isinstance(data, dict) and data.get("error"):
            raise RuntimeError(f"HF API error: {data['error']}")
        if isinstance(data, list) and len(data) > 0 and "generated_text" in data[0]:
            return data[0]["generated_text"].strip()
        return str(data)

    return _run_with_retries(_call)

def generate_answer(provider: str, prompt: str, model: str, temperature: float = 0.2, max_tokens: int = 500) -> str:
    allowed_models = _allowed_models_from_selection()
    if allowed_models and model not in allowed_models:
        raise ValueError(
            f"Model '{model}' is not allowed. Use only models defined in Section 3.1: {sorted(allowed_models)}"
        )

    provider = provider.strip().lower()
    if provider in {"gemini_model", "gemini"}:
        return generate_gemini(prompt, model=model, temperature=temperature, max_tokens=max_tokens)
    if provider == "openai":
        return generate_openai(prompt, model=model, temperature=temperature, max_tokens=max_tokens)
    if provider == "anthropic":
        return generate_anthropic(prompt, model=model, temperature=temperature, max_tokens=max_tokens)
    if provider == "huggingface":
        return generate_huggingface(prompt, model=model, temperature=temperature, max_tokens=max_tokens)
    raise ValueError("Unknown provider")

## 7) Define Benchmark Questions (Increasing Difficulty)

The benchmark set is intentionally designed as a difficulty ladder.

Why this is pedagogically important:
- Easy questions test basic retrieval quality.
- Medium questions test aggregation and reasoning over multiple retrieved records.
- Hard questions test uncertainty handling and resistance to false assumptions.

Question types included:
- Basic retrieval.
- Structured reasoning.
- Ambiguous interpretation.
- False-premise challenge.
- Negative control (information likely missing from context).

Good evaluation practice:
- Keep question wording fixed across models to preserve fairness.

In [22]:
benchmark_questions = [
    {
        "question_id": "q1",
        "difficulty": 1,
        "category": "basic_retrieval",
        "question": "List all campaigns related to earthquakes.",
    },
    {
        "question_id": "q2",
        "difficulty": 2,
        "category": "structured_reasoning",
        "question": "What regions appear most frequently across campaigns?",
    },
    {
        "question_id": "q3",
        "difficulty": 3,
        "category": "ambiguous",
        "question": "Describe how Swiss Solidarity supports children in its campaigns.",
    },
    {
        "question_id": "q4",
        "difficulty": 4,
        "category": "false_assumptions",
        "question": "Which campaign lasted the longest?",
    },
    {
        "question_id": "q5",
        "difficulty": 5,
        "category": "negative_control",
        "question": "What percentage of funds went to administrative costs?",
    },
]

pd.DataFrame(benchmark_questions)

,question_id,difficulty,category,question
0,q1,1,basic_retrieval,List all campaigns related to earthquakes.
1,q2,2,structured_reasoning,What regions appear most frequently across cam...
2,q3,3,ambiguous,Describe how Swiss Solidarity supports childre...
3,q4,4,false_assumptions,Which campaign lasted the longest?
4,q5,5,negative_control,What percentage of funds went to administrativ...


## 8) Validate Selected Models Against Available Secrets

This section does **not** select models.

Instead, it validates the models you already selected in Section 3.1, checks whether required API keys are available, and activates only valid configurations for execution.

In [59]:
if "BENCHMARK_MODELS" not in globals():
    raise RuntimeError("BENCHMARK_MODELS is not defined. Run Section 3.1 first.")

if len(BENCHMARK_MODELS) != 2:
    raise ValueError("Please select exactly 2 answer models in Section 3.1.")

active_models = []
for m in BENCHMARK_MODELS:
    api_key_name = m.get("api_key_name", "")
    if API_KEYS.get(api_key_name):
        active_models.append(m)
    else:
        print(f"Skipping {m.get('llm_id', 'unknown')} (missing {api_key_name}).")

if len(active_models) != 2:
    raise RuntimeError(
        "Exactly 2 answer models are required, but one or more selected models are missing API keys."
    )

active_providers = sorted({m["provider"] for m in active_models})

print("Active providers:", active_providers)
print("Active models:", [f"{m['llm_id']} ({m['model']})" for m in active_models])

Active providers: ['gemini']
Active models: ['gemini_answerer_a (gemma-4-31b-it)', 'gemini_answerer_b (gemma-4-26b-a4b-it)']


## 9) RAG Prompting and Batch Answer Generation

This is the main experiment loop.

For each question and for each active model, the notebook:
- Retrieves top-k evidence passages.
- Builds one grounded prompt containing the same context for all models.
- Requests an answer from the current model.
- Stores answer text, retrieval evidence, and any runtime error.

Why this design is fair:
- All models receive the same question and the same retrieved context.
- Differences in quality are more likely due to model behavior, not retrieval differences.

Saved outputs:
- JSONL and CSV answer logs for later auditing and error analysis.

In [53]:
import google.generativeai as genai

api_key = API_KEYS.get("GEMINI_API_KEY", "")
if not api_key:
    raise ValueError("GEMINI_API_KEY is missing.")

genai.configure(api_key=api_key)

gemini_models = sorted(list(genai.list_models()), key=lambda m: getattr(m, "name", ""))

print(f"Total models available: {len(gemini_models)}\n")
for m in gemini_models:
    print(
        f"{m.name} | "
        f"display_name={getattr(m, 'display_name', '')} | "
        f"generation_methods={getattr(m, 'supported_generation_methods', [])}"
    )

Total models available: 54

models/antigravity-preview-05-2026 | display_name=Antigravity Agent Preview | generation_methods=['generateContent', 'countTokens']
models/aqa | display_name=Model that performs Attributed Question Answering. | generation_methods=['generateAnswer']
models/deep-research-max-preview-04-2026 | display_name=Deep Research Max Preview (Apr-21-2026) | generation_methods=['generateContent', 'countTokens']
models/deep-research-preview-04-2026 | display_name=Deep Research Preview (Apr-21-2026) | generation_methods=['generateContent', 'countTokens']
models/deep-research-pro-preview-12-2025 | display_name=Deep Research Pro Preview (Dec-12-2025) | generation_methods=['generateContent', 'countTokens']
models/gemini-2.0-flash | display_name=Gemini 2.0 Flash | generation_methods=['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
models/gemini-2.0-flash-001 | display_name=Gemini 2.0 Flash 001 | generation_methods=['generateContent', 'countToken

In [56]:
# Quick connectivity/sanity test for each active answer model
test_prompt = "Reply with exactly: OK"

test_rows = []
for m in active_models:
    started = time.perf_counter()
    answer_text = ""
    error_text = ""

    try:
        answer_text = generate_answer(
            provider=m["provider"],
            model=m["model"],
            prompt=test_prompt,
            temperature=0.0,
            max_tokens=20,
        )
    except Exception as e:
        error_text = f"{type(e).__name__}: {e}"

    elapsed_s = time.perf_counter() - started
    test_rows.append({
        "llm_id": m["llm_id"],
        "provider": m["provider"],
        "model": m["model"],
        "ok": error_text == "" and bool(str(answer_text).strip()),
        "latency_s": round(elapsed_s, 2),
        "answer_preview": (str(answer_text).strip()[:120] if answer_text else ""),
        "error": error_text,
    })

model_test_df = pd.DataFrame(test_rows)
display(model_test_df)

n_ok = int(model_test_df["ok"].sum())
print(f"Working models: {n_ok}/{len(model_test_df)}")

Gemini model failed: gemma-4-26b-it -> NotFound
Gemini model failed: gemini-2.5-flash -> ValueError
Gemini model failed: gemini-1.5-flash -> NotFound


,llm_id,provider,model,ok,latency_s,answer_preview,error
0,gemini_answerer_a,gemini,gemma-4-31b-it,True,1.29,"The user wants me to reply with exactly ""OK"".\...",
1,gemini_answerer_b,gemini,gemma-4-26b-it,False,0.87,,RuntimeError: Gemini request failed for models...


Working models: 1/2


In [61]:
def build_context(retrieved_df: pd.DataFrame, max_chars: int = 7000) -> str:
    chunks = []
    total = 0
    for _, row in retrieved_df.iterrows():
        piece = (
            f"Title: {row['title']}\n"
            f"URL: {row['url']}\n"
            f"Instance ID: {row['campaign_instance_id']}\n"
            f"Similarity: {row['similarity']:.4f}\n"
            f"Text: {row['campaign_text']}\n"
        )
        if total + len(piece) > max_chars:
            break
        chunks.append(piece)
        total += len(piece)
    return "\n---\n".join(chunks)

def build_rag_prompt(question: str, context: str) -> str:
    return f"""You are a data analyst helping Swiss Solidarity to analyze campaign data.
Answer ONLY from the provided campaign context.
If the answer is not available in the context, clearly say that and do not invent values.

Question: {question}

Campaign context:
{context}

Return:
1) A concise answer.
2) Bullet points with campaign titles and URLs used as evidence.
"""

rows = []
for q in tqdm(benchmark_questions, desc="Questions"):
    retrieved = retrieve_top_k(q["question"], k=TOP_K)
    context = build_context(retrieved, max_chars=MAX_CONTEXT_CHARS)
    prompt = build_rag_prompt(q["question"], context)

    for m in active_models:
        answer = ""
        error = ""
        try:
            answer = generate_answer(
                provider=m["provider"],
                model=m["model"],
                prompt=prompt,
                temperature=TEMPERATURE,
                max_tokens=MAX_OUTPUT_TOKENS,
            )
        except Exception as e:
            error = f"{type(e).__name__}: {e}"

        rows.append({
            "question_id": q["question_id"],
            "difficulty": q["difficulty"],
            "category": q["category"],
            "question": q["question"],
            "llm_id": m["llm_id"],
            "provider": m["provider"],
            "model": m["model"],
            "answer": answer,
            "error": error,
            "context": context,
            "retrieved_titles": retrieved["title"].tolist(),
            "retrieved_urls": retrieved["url"].tolist(),
        })

answers_df = pd.DataFrame(rows)
answers_df.to_json("rag_benchmark_answers.jsonl", orient="records", lines=True, force_ascii=False)
answers_df.to_csv("rag_benchmark_answers.csv", index=False)

print("Saved: rag_benchmark_answers.jsonl")
print("Saved: rag_benchmark_answers.csv")
answers_df[["question_id", "llm_id", "error"]].head(20)

Questions:   0%|          | 0/5 [00:00<?, ?it/s]

Saved: rag_benchmark_answers.jsonl
Saved: rag_benchmark_answers.csv


,question_id,llm_id,error
0,q1,gemini_answerer_a,
1,q1,gemini_answerer_b,
2,q2,gemini_answerer_a,
3,q2,gemini_answerer_b,
4,q3,gemini_answerer_a,
5,q3,gemini_answerer_b,
6,q4,gemini_answerer_a,
7,q4,gemini_answerer_b,
8,q5,gemini_answerer_a,
9,q5,gemini_answerer_b,


## 10) LLM-as-Judge Rubric

This section introduces the judge design and then runs the judge loop in the following code cell.

Metrics (0 to 5 each):
- `factuality`: Is the answer correct according to retrieved context?
- `conciseness`: Is it brief and efficient?
- `groundedness`: Does it avoid claims not supported by context?
- `completeness`: Does it cover what the question asks?
- `uncertainty_handling`: Does it clearly say when evidence is missing?

What the next code cell **defines**:
- Judge setup validation from `JUDGE_MODEL_CONFIG` (selected in Section 3.1).
- Judge prompt template.
- JSON parsing and weighted scoring helpers.

What the next code cell **executes**:
- Iterates through `answers_df` and evaluates each answer.
- Writes judged outputs to JSONL/CSV.

Scoring design:
- A weighted overall score combines all metrics.
- Higher weight on factuality and groundedness reflects RAG priorities.

Important caution for students:
- The judge is still a model and can be biased or inconsistent.
- If stakes are high, combine LLM judging with human review or deterministic checks.

In [65]:
if "JUDGE_MODEL_CONFIG" not in globals():
    raise RuntimeError("JUDGE_MODEL_CONFIG is not defined. Run Section 3.1 first.")

JUDGE_PROVIDER = str(JUDGE_MODEL_CONFIG.get("provider", "")).strip().lower()
JUDGE_MODEL = str(JUDGE_MODEL_CONFIG.get("model", "")).strip()
JUDGE_API_KEY_NAME = str(JUDGE_MODEL_CONFIG.get("api_key_name", "")).strip()

if not JUDGE_PROVIDER or not JUDGE_MODEL or not JUDGE_API_KEY_NAME:
    raise ValueError("JUDGE_MODEL_CONFIG must include provider, model, and api_key_name.")

if JUDGE_PROVIDER not in active_providers:
    raise RuntimeError(
        f"Judge provider '{JUDGE_PROVIDER}' is not active. Check selected models and API keys."
    )

if not API_KEYS.get(JUDGE_API_KEY_NAME):
    raise RuntimeError(f"Missing judge API key: {JUDGE_API_KEY_NAME}")

print("Judge provider:", JUDGE_PROVIDER)
print("Judge model:", JUDGE_MODEL)

judge_weights = {
    "factuality": 0.35,
    "conciseness": 0.15,
    "groundedness": 0.25,
    "completeness": 0.15,
    "uncertainty_handling": 0.10,
}

def build_judge_prompt(question: str, context: str, answer: str) -> str:
    return f"""You are an impartial evaluator of RAG answers.
Given the QUESTION, RETRIEVAL CONTEXT, and MODEL ANSWER, score the answer from 0 to 5 on each metric:
1) factuality
2) conciseness
3) groundedness
4) completeness
5) uncertainty_handling

Rules:
- Use only the provided context as evidence.
- Penalize fabricated facts.
- Reward explicit acknowledgement when answer is not in context.
- Keep rationale short.

QUESTION:
{question}

RETRIEVAL CONTEXT:
{context}

MODEL ANSWER:
{answer}

Return strict JSON with this schema only:
{{
  "factuality": number,
  "conciseness": number,
  "groundedness": number,
  "completeness": number,
  "uncertainty_handling": number,
  "rationale": string
}}
"""

def extract_json_obj(text: str) -> dict:
    # Handles responses that wrap JSON in markdown fences or extra text.
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\n", "", text)
        text = re.sub(r"\n```$", "", text)

    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        text = text[first:last + 1]

    return json.loads(text)

def score_weighted(row: dict) -> float:
    total = 0.0
    for metric, w in judge_weights.items():
        total += float(row.get(metric, 0.0)) * w
    return total

judge_rows = []
for _, r in tqdm(answers_df.iterrows(), total=len(answers_df), desc="Judging"):
    if r.get("error"):
        judge_rows.append({
            "question_id": r["question_id"],
            "llm_id": r["llm_id"],
            "judge_error": f"Skipped due to model error: {r['error']}",
            "factuality": 0.0,
            "conciseness": 0.0,
            "groundedness": 0.0,
            "completeness": 0.0,
            "uncertainty_handling": 0.0,
            "rationale": "Candidate model failed to produce an answer.",
            "overall": 0.0,
        })
        continue

    prompt = build_judge_prompt(r["question"], r["context"], r["answer"])
    judge_error = ""
    judge_raw = ""
    parsed = {}

    try:
        judge_raw = generate_answer(
            provider=JUDGE_PROVIDER,
            model=JUDGE_MODEL,
            prompt=prompt,
            temperature=0.0,
            max_tokens=700,
        )
        parsed = extract_json_obj(judge_raw)
    except Exception as e:
        judge_error = f"{type(e).__name__}: {e}"

    row_out = {
        "question_id": r["question_id"],
        "llm_id": r["llm_id"],
        "judge_error": judge_error,
        "factuality": float(parsed.get("factuality", 0.0)) if not judge_error else 0.0,
        "conciseness": float(parsed.get("conciseness", 0.0)) if not judge_error else 0.0,
        "groundedness": float(parsed.get("groundedness", 0.0)) if not judge_error else 0.0,
        "completeness": float(parsed.get("completeness", 0.0)) if not judge_error else 0.0,
        "uncertainty_handling": float(parsed.get("uncertainty_handling", 0.0)) if not judge_error else 0.0,
        "rationale": parsed.get("rationale", "") if not judge_error else "Judge failed.",
        "judge_raw": judge_raw,
    }
    row_out["overall"] = score_weighted(row_out)
    judge_rows.append(row_out)

judge_df = pd.DataFrame(judge_rows)
eval_df = answers_df.merge(judge_df, on=["question_id", "llm_id"], how="left")

eval_df.to_json("rag_benchmark_judged.jsonl", orient="records", lines=True, force_ascii=False)
eval_df.to_csv("rag_benchmark_judged.csv", index=False)

print("Saved: rag_benchmark_judged.jsonl")
print("Saved: rag_benchmark_judged.csv")
eval_df[["question_id", "llm_id", "overall", "judge_error"]].head(20)

Judge provider: gemini
Judge model: gemini-3.1-flash-lite


Judging:   0%|          | 0/10 [00:00<?, ?it/s]

Saved: rag_benchmark_judged.jsonl
Saved: rag_benchmark_judged.csv


,question_id,llm_id,overall,judge_error
0,q1,gemini_answerer_a,4.70,
1,q1,gemini_answerer_b,4.70,
2,q2,gemini_answerer_a,4.85,
3,q2,gemini_answerer_b,5.00,
4,q3,gemini_answerer_a,5.00,
5,q3,gemini_answerer_b,5.00,
6,q4,gemini_answerer_a,4.55,
7,q4,gemini_answerer_b,4.25,
8,q5,gemini_answerer_a,5.00,
9,q5,gemini_answerer_b,5.00,


## 11) Determine Winner per Question and Aggregate Ranking

This final section turns raw scores into interpretable benchmark results.

Two complementary views are produced:
- Per-question winner: which model scored best on each question.
- Global leaderboard: average metric scores across all questions.

How students should interpret results:
- Per-question winners show strengths on specific reasoning patterns.
- Leaderboard averages show overall robustness.
- Inspect rationales and errors, not just final numeric rank.

Recommended classroom discussion:
- Which models perform better on false-premise and negative-control questions?
- Does concise writing correlate with factual quality?
- Are there cases where judge confidence seems questionable?

In [66]:
winners = (
    eval_df.sort_values(["question_id", "overall"], ascending=[True, False])
    .groupby("question_id", as_index=False)
    .first()[["question_id", "question", "llm_id", "provider", "model", "overall", "rationale"]]
)

leaderboard = (
    eval_df.groupby(["llm_id", "provider", "model"], as_index=False)[[
        "factuality",
        "conciseness",
        "groundedness",
        "completeness",
        "uncertainty_handling",
        "overall",
    ]]
    .mean()
    .sort_values("overall", ascending=False)
)

winners.to_csv("rag_benchmark_winners_per_question.csv", index=False)
leaderboard.to_csv("rag_benchmark_leaderboard.csv", index=False)

print("Saved: rag_benchmark_winners_per_question.csv")
print("Saved: rag_benchmark_leaderboard.csv")

display(winners)
display(leaderboard)

Saved: rag_benchmark_winners_per_question.csv
Saved: rag_benchmark_leaderboard.csv


,question_id,question,llm_id,provider,model,overall,rationale
0,q1,List all campaigns related to earthquakes.,gemini_answerer_a,gemini,gemma-4-31b-it,4.70,The model correctly identified the relevant ca...
1,q2,What regions appear most frequently across cam...,gemini_answerer_b,gemini,gemma-4-26b-a4b-it,5.00,The model correctly identified the most freque...
2,q3,Describe how Swiss Solidarity supports childre...,gemini_answerer_a,gemini,gemma-4-31b-it,5.00,The model correctly identified that the provid...
3,q4,Which campaign lasted the longest?,gemini_answerer_a,gemini,gemma-4-31b-it,4.55,The model correctly identified the campaign wi...
4,q5,What percentage of funds went to administrativ...,gemini_answerer_a,gemini,gemma-4-31b-it,5.00,The model correctly identified that the provid...


,llm_id,provider,model,factuality,conciseness,groundedness,completeness,uncertainty_handling,overall
0,gemini_answerer_a,gemini,gemma-4-31b-it,5.0,3.8,5.0,5.0,5.0,4.82
1,gemini_answerer_b,gemini,gemma-4-26b-a4b-it,5.0,3.8,5.0,4.8,5.0,4.79


## 12) Visual Comparison and Statistical Significance

This section helps you interpret model quality beyond raw tables.

What you will get:
- Comparative graphs per metric (dimension) and per question.
- Overall score comparisons per question.
- Statistical tests to check whether observed differences are likely meaningful.

Pedagogical note:
- With very small samples (few questions), p-values are unstable.
- Read p-values together with effect sizes and score plots.

In [ ]:
try:
    import plotly.express as px
except Exception as e:
    raise ImportError("Plotly is required for visualization. Run Cell 3 install command first.") from e

if "eval_df" not in globals():
    eval_df = pd.read_csv("rag_benchmark_judged.csv")

if "benchmark_questions" in globals():
    q_order = [q["question_id"] for q in benchmark_questions]
else:
    q_order = sorted(eval_df["question_id"].unique())

metrics = ["factuality", "conciseness", "groundedness", "completeness", "uncertainty_handling"]
score_cols = metrics + ["overall"]

plot_df = eval_df[["question_id", "llm_id", *score_cols]].copy()
plot_df["question_id"] = pd.Categorical(plot_df["question_id"], categories=q_order, ordered=True)

long_dim = plot_df.melt(
    id_vars=["question_id", "llm_id"],
    value_vars=metrics,
    var_name="dimension",
    value_name="score",
)

fig_dim = px.bar(
    long_dim.sort_values(["dimension", "question_id", "llm_id"]),
    x="question_id",
    y="score",
    color="llm_id",
    barmode="group",
    facet_col="dimension",
    facet_col_wrap=3,
    title="Model Comparison by Dimension and Question",
    labels={"question_id": "Question", "llm_id": "Model", "score": "Score (0-5)", "dimension": "Dimension"},
)
fig_dim.update_yaxes(range=[0, 5])
fig_dim.update_layout(height=900)
fig_dim.show()

fig_overall = px.bar(
    plot_df.sort_values(["question_id", "llm_id"]),
    x="question_id",
    y="overall",
    color="llm_id",
    barmode="group",
    title="Overall Score per Question",
    labels={"question_id": "Question", "llm_id": "Model", "overall": "Overall Score (0-5)"},
)
fig_overall.update_yaxes(range=[0, 5])
fig_overall.show()

heat_df = plot_df.pivot_table(index="llm_id", columns="question_id", values="overall", aggfunc="mean")
heat_df = heat_df.reindex(columns=q_order)
fig_heat = px.imshow(
    heat_df,
    text_auto=".2f",
    color_continuous_scale="YlGnBu",
    zmin=0,
    zmax=5,
    aspect="auto",
    title="Overall Score Heatmap (Model x Question)",
    labels={"x": "Question", "y": "Model", "color": "Overall"},
)
fig_heat.show()

summary_means = (
    eval_df.groupby("llm_id", as_index=False)[score_cols]
    .mean()
    .sort_values("overall", ascending=False)
    .reset_index(drop=True)
)
display(summary_means)

In [ ]:
import math
import itertools

def two_sided_sign_test_p(wins: int, losses: int) -> float:
    n = wins + losses
    if n == 0:
        return float("nan")
    k = min(wins, losses)
    tail = sum(math.comb(n, i) for i in range(0, k + 1)) / (2 ** n)
    return min(1.0, 2 * tail)

def paired_signflip_pvalue(diffs: np.ndarray) -> float:
    diffs = np.asarray(diffs, dtype=float)
    diffs = diffs[~np.isnan(diffs)]
    diffs = diffs[diffs != 0]
    n = len(diffs)
    if n == 0:
        return float("nan")
    abs_vals = np.abs(diffs)
    observed = abs(np.sum(diffs))
    extreme = 0
    total = 2 ** n
    for signs in itertools.product([-1.0, 1.0], repeat=n):
        stat = abs(np.sum(abs_vals * np.array(signs)))
        if stat >= observed - 1e-12:
            extreme += 1
    return extreme / total

def cohens_dz(diffs: np.ndarray) -> float:
    diffs = np.asarray(diffs, dtype=float)
    diffs = diffs[~np.isnan(diffs)]
    if len(diffs) < 2:
        return float("nan")
    sd = diffs.std(ddof=1)
    if sd == 0:
        return float("nan")
    return diffs.mean() / sd

try:
    from scipy.stats import wilcoxon  # Optional; sign-flip/sign test still run without scipy
    has_scipy = True
except Exception:
    has_scipy = False

score_cols = ["factuality", "conciseness", "groundedness", "completeness", "uncertainty_handling", "overall"]
models = sorted(eval_df["llm_id"].unique())
pairs = list(itertools.combinations(models, 2))

if not pairs:
    raise RuntimeError("Need at least two models in eval_df for significance testing.")

rows = []
for m1, m2 in pairs:
    left = eval_df[eval_df["llm_id"] == m1][["question_id", *score_cols]].rename(
        columns={c: f"{c}_m1" for c in score_cols}
    )
    right = eval_df[eval_df["llm_id"] == m2][["question_id", *score_cols]].rename(
        columns={c: f"{c}_m2" for c in score_cols}
    )
    pair_df = left.merge(right, on="question_id", how="inner")

    for metric in score_cols:
        a = pair_df[f"{metric}_m1"].astype(float).to_numpy()
        b = pair_df[f"{metric}_m2"].astype(float).to_numpy()
        diffs = a - b

        wins = int(np.sum(diffs > 0))
        losses = int(np.sum(diffs < 0))
        ties = int(np.sum(diffs == 0))

        sign_p = two_sided_sign_test_p(wins, losses)
        sf_p = paired_signflip_pvalue(diffs)
        dz = cohens_dz(diffs)

        wilcoxon_p = float("nan")
        if has_scipy and (wins + losses) > 0:
            try:
                wilcoxon_p = float(wilcoxon(a, b, zero_method="wilcox", alternative="two-sided").pvalue)
            except Exception:
                wilcoxon_p = float("nan")

        rows.append({
            "model_a": m1,
            "model_b": m2,
            "metric": metric,
            "n_questions": int(len(pair_df)),
            "mean_a": float(np.mean(a)),
            "mean_b": float(np.mean(b)),
            "mean_diff_a_minus_b": float(np.mean(diffs)),
            "wins_a": wins,
            "wins_b": losses,
            "ties": ties,
            "cohens_dz": dz,
            "p_sign_test": sign_p,
            "p_sign_flip_exact": sf_p,
            "p_wilcoxon": wilcoxon_p,
        })

stats_df = pd.DataFrame(rows).sort_values(["metric", "model_a", "model_b"]).reset_index(drop=True)
stats_df.to_csv("rag_benchmark_significance_tests.csv", index=False)
print("Saved: rag_benchmark_significance_tests.csv")

display(stats_df)

overall_stats = stats_df[stats_df["metric"] == "overall"].copy().sort_values("p_sign_flip_exact")
print("Overall metric significance summary:")
display(overall_stats)